In [4]:
## 项目目标：评估DeepSeek的代码能力
## 项目步骤：
# （1）根据测试集，利用DeepSeek的API，得到Deepseek的推理结果
# （2）对推理结果做处理:1.拆分测试用例；2.处理模型的输出，得到模型的函数实现体
# （3）执行代码，测试测试用例，并统计得分


In [4]:
import json
with open("data/code_test_result_0629_split/problem_code_test_result_0629_split_model1.jsonl", encoding="utf-8") as f:
    for line in f:
        print(json.loads(line))

{'task_id': 'List_2_1', 'import_code': 'from typing import List\n', 'prompt': 'from typing import List \n\ndef closed_lights(n: int, m: int) -> List[int]: \n    """ 有n盏灯，从1到n顺序编号，初始状态都为开启状态。有m个人，第一个人会将1的倍数的灯关闭，第二个人会将2的倍数的灯打开，第三个人会将3的倍数的灯作相反操作(改变灯的状态为相反状态)，以此类推。问：第m个人操作后，哪几盏灯是关闭的，按升序结果输出。\n    >>> closed_lights(10, 10) \n    [1, 4, 9]\n    """', 'canonical_solution': None, 'test': 'def check(candidate):\n    assert candidate(10, 10) == [1, 4, 9]', 'entry_point': 'closed_lights', 'origin_model_response': 'from typing import List\n\ndef closed_lights(n: int, m: int) -> List[int]:\n    lights = [False] * (n + 1)  # False表示灯关闭，True表示灯开启（初始状态应为开启）\n    \n    # 初始状态都为开启（True），所以需要先全部打开\n    for i in range(1, n + 1):\n        lights[i] = True\n    \n    for person in range(1, m + 1):\n        for light in range(person, n + 1, person):\n            if person == 1:\n                lights[light] = False  # 第一个人关闭\n            elif person == 2:\n                lights[light] = True  # 第二个人打开\n   

In [10]:
from openai import OpenAI

client = OpenAI(api_key="****************", base_url="https://api.deepseek.com")

In [5]:
prompt = '''代码生成任务
1.任务名称：
根据需求描述和测试用例来生成代码。

2.任务描述：
需求描述定义了需要生成的代码的用途和要求；提供的测试用例包括了代码的输入参数列表、期望的输出，用于测试你生成的代码。
你生成的所有代码需要包含必要的库导入步骤。不允许更改方法的名称和已经给定的形式参数的名称和类型。
除了定义函数的功能体和必要的包导入步骤，你生成的代码不应该包括任何多余内容。你生成的代码需要符合Python代码格式要求，要能正确运行。

3.生成的代码的片段必须符合下面的格式要求：
【代码开始】
your code here
【代码结束】

比如：
【代码开始】
from typing import List

def has_xxx_elements(numbers: List[float], threshold: float) -> bool:
    for i in range(len(numbers)):
    return False
【代码结束】

4. 需求描述和测试用例
<需求描述和测试用例--开始>
{question_turns_1}
<需求描述和测试用例--结束>
'''

In [11]:
def inference_one(prompt):
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "user", "content": prompt},
            ],
            stream=False
        )
    except Exception as e:
        print(f"exception is {e}")
        return ""
    
    return  response.choices[0].message.content

In [12]:
code_question = '''from typing import List 

def closed_lights(n: int, m: int) -> List[int]: 
    """ 有n盏灯，从1到n顺序编号，初始状态都为开启状态。有m个人，第一个人会将1的倍数的灯关闭，第二个人会将2的倍数的灯打开，第三个人会将3的倍数的灯作相反操作(改变灯的状态为相反状态)，以此类推。问：第m个人操作后，哪几盏灯是关闭的，按升序结果输出。
    >>> closed_lights(10, 10) 
    [1, 4, 9]
    """
'''
query = prompt.replace("{question_turns_1}", code_question)

res = inference_one(query)
res 

'【代码开始】\nfrom typing import List\n\ndef closed_lights(n: int, m: int) -> List[int]:\n    lights = [True] * (n + 1)  # True表示灯开启，False表示关闭\n    for person in range(1, m + 1):\n        for light in range(person, n + 1, person):\n            if person == 1:\n                lights[light] = False\n            elif person == 2:\n                lights[light] = True\n            else:\n                lights[light] = not lights[light]\n    result = [i for i in range(1, n + 1) if not lights[i]]\n    return result\n【代码结束】'

In [13]:
import pandas as pd
from tqdm import tqdm

#读取测试集
code_test_data = pd.read_excel("data/model_response/code_test.xlsx")

model_answer_model1_turns_1 = []
for i in tqdm(range(len(code_test_data))):
    #读取代码问题
    question = code_test_data.iloc[i]["question_turns_1"]
    print(f"start to generate answer, question is {question}")
    #构造指令
    query = prompt.replace("{question_turns_1}", code_question)
    #利用大模型得到答案
    answer = inference_one(query)
    print(f"start to generate answer, answer is {answer}")
    model_answer_model1_turns_1.append(answer)

code_test_data["model_answer_model1_turns_1"] = model_answer_model1_turns_1

code_test_data.to_excel("data/model_response/code_test_result.xlsx")

  0%|          | 0/101 [00:00<?, ?it/s]

start to generate answer, question is from typing import List 

def closed_lights(n: int, m: int) -> List[int]: 
    """ 有n盏灯，从1到n顺序编号，初始状态都为开启状态。有m个人，第一个人会将1的倍数的灯关闭，第二个人会将2的倍数的灯打开，第三个人会将3的倍数的灯作相反操作(改变灯的状态为相反状态)，以此类推。问：第m个人操作后，哪几盏灯是关闭的，按升序结果输出。
    >>> closed_lights(10, 10) 
    [1, 4, 9]
    """


  1%|          | 1/101 [00:08<14:47,  8.88s/it]

start to generate answer, answer is 【代码开始】
from typing import List

def closed_lights(n: int, m: int) -> List[int]:
    lights = [True] * (n + 1)  # True表示开启，False表示关闭
    for person in range(1, m + 1):
        for light in range(person, n + 1, person):
            if person == 1:
                lights[light] = False
            elif person == 2:
                lights[light] = True
            else:
                lights[light] = not lights[light]
    result = [i for i in range(1, n + 1) if not lights[i]]
    return result
【代码结束】
start to generate answer, question is from typing import List 

def suitable_matrix(mat: List[List[int]]) -> bool: 
    """ 给定n×n由0和1组成的矩阵，如果矩阵的每一行和每一列的1的数量都是偶数，则认为是符合条件的矩阵。现要求检测矩阵，最多只能改变矩阵中的一个元素，是否为符合条件的矩阵。改变矩阵元素是指0变成1或1变成0。
    >>> suitable_matrix([[1,0,1,0],
                    [0,0,0,0],
                    [1,1,1,1],
                    [0,1,0,1]]) 
    True
    """


  2%|▏         | 2/101 [00:17<14:05,  8.54s/it]

start to generate answer, answer is 【代码开始】
from typing import List

def closed_lights(n: int, m: int) -> List[int]:
    lights = [True] * (n + 1)  # True表示开启，False表示关闭
    for person in range(1, m + 1):
        for light in range(person, n + 1, person):
            if person == 1:
                lights[light] = False
            elif person == 2:
                lights[light] = True
            else:
                lights[light] = not lights[light]
    result = [i for i in range(1, n + 1) if not lights[i]]
    return result
【代码结束】
start to generate answer, question is from typing import List

def similar_image(mat1: List[List[int]], mat2: List[List[int]]) -> float:
    """ 给定两个相同大小的矩阵，矩阵只有0和1的值，求它们的相似度。两幅图像的相似度定义为相同像素点数占总像素点数的百分比。相似度以百分比给出，精确到小数点后两位。
    >>> similar_image([[1,0,1],
                        [0,0,1],
                        [1,1,0]],
                        [[1,1,0],
                        [0,0,1],
                        [0,0,1]])
    44.44
    """


  3%|▎         | 3/101 [00:24<13:23,  8.20s/it]

start to generate answer, answer is 【代码开始】
from typing import List

def closed_lights(n: int, m: int) -> List[int]:
    lights = [True] * (n + 1)  # True表示开启，False表示关闭
    for person in range(1, m + 1):
        for light in range(person, n + 1, person):
            if person == 1:
                lights[light] = False
            elif person == 2:
                lights[light] = True
            else:
                lights[light] = not lights[light]
    return [i for i in range(1, n + 1) if not lights[i]]
【代码结束】
start to generate answer, question is from typing import List

def magical_matrix(mat: List[List[int]]) -> bool:
    """ 判断n×n矩阵中每行元素之和，每列元素之和及每个对角线上元素之和均相等。均相等返回True，否则返回False
    >>> magical_matrix([[17,24,1,8,15],
                        [23,5,7,14,16],
                        [4,6,13,20,22],
                        [10,12,19,21,3],
                        [11,18,25,2,9]])
    True
    """


  3%|▎         | 3/101 [00:36<19:39, 12.03s/it]


KeyboardInterrupt: 

In [ ]:



def closed_lights(n: int, m: int) -> List[int]
    lights = [False] * (n + 1)  # False表示灯关闭，True表示灯开启（初始状态应为开启）
    
    # 初始状态都为开启（True），所以需要先全部打开
    for i in range(1, n + 1):
        lights[i] = True
    
    for person in range(1, m + 1):
        for light in range(person, n + 1, person):
            if person == 1:
                lights[light] = False  # 第一个人关闭
            elif person == 2:
                lights[light] = True  # 第二个人打开
            else:
                lights[light] = not lights[light]  # 其他人切换状态
    
    result = [i for i in range(1, n + 1) if not lights[i]]
    return result
assert closed_lights(10, 10) == [1, 4, 9]